In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Φόρτωση Δεδομένων 
train_path = 'processed/train_final_scaled.csv'
test_path = 'processed/test_final_scaled.csv'

# Διαβάζουμε τα columns και φτιάχνουμε τα types
columns = pd.read_csv(train_path, nrows=0).columns.tolist()
features = [col for col in columns if col != "Label"]
dtypes = {col: "float32" for col in features}
dtypes["Label"] = "int16"

train_df = pd.read_csv(train_path, dtype=dtypes)
test_df = pd.read_csv(test_path, dtype=dtypes)

# Διαχωρισμός σε Features (X)
X_train = train_df[features].to_numpy(dtype=np.float32)
X_test = test_df[features].to_numpy(dtype=np.float32)

# Μετατροπή σε Labels (0=Benign, 1=Attack)
BENIGN_LABEL = 1

# Ορίζουμε απευθείας τα binary labels (ό,τι δεν είναι 1, γίνεται 1, δηλαδή Attack)
y_train_bin = (train_df["Label"].to_numpy() != BENIGN_LABEL).astype(int)
y_test_bin = (test_df["Label"].to_numpy() != BENIGN_LABEL).astype(int)

print("=== DATASET ===")
print("Training:", X_train.shape)
print("Testing:", X_test.shape)
print("Features:", len(features))

print("\nTraining classes:", np.bincount(y_train_bin))
print("Testing classes:", np.bincount(y_test_bin))

In [ ]:
# --- SUPERVISED MODEL: Random Forest ---

# Κοινή συνάρτηση αξιολόγησης
def evaluate_model(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "FPR": fp / (fp + tn)
    }

    print(f"\n=== {name} RESULTS ===")

    for metric, value in metrics.items():
        print(f"{metric}: {value:.6f}")

    print("\nConfusion Matrix:")
    print(cm)

    return metrics


# Supervised Model: Random Forest
rf = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train_bin)
rf_preds = rf.predict(X_test)

rf_metrics = evaluate_model(
    "Random Forest", y_test_bin, rf_preds
)


In [ ]:

# --- UNSUPERVISED MODEL: K-Means ---

kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init="auto"
)

# Εκπαίδευση χωρίς Labels
train_clusters = kmeans.fit_predict(X_train)

# Το μικρότερο cluster θεωρείται υποψήφιο Attack
cluster_sizes = np.bincount(train_clusters, minlength=2)
attack_cluster = int(np.argmin(cluster_sizes))

# Πρόβλεψη στο Testing Set
test_clusters = kmeans.predict(X_test)
kmeans_preds = (test_clusters == attack_cluster).astype(int)

# Αξιολόγηση
kmeans_metrics = evaluate_model(
    "K-Means", y_test_bin, kmeans_preds
)

print("\nTraining cluster sizes:", cluster_sizes)
print("Cluster assigned to Attack:", attack_cluster)

# Διαγνωστική ανάλυση των clusters
print("\nClass distribution per TEST cluster:")
print(pd.crosstab(
    test_clusters,
    y_test_bin,
    normalize="index"
).round(4))


In [ ]:

import time
from pathlib import Path

# Κοινή συνάρτηση μέτρησης inference
def benchmark_model(model, X_test, name, batch_size=512):
    print(f"=== {name} INFERENCE ===")

    # Warm-up
    for _ in range(100):
        model.predict(X_test[:1])

    for _ in range(20):
        model.predict(X_test[:batch_size])

    # 1. Single-flow latency
    single_times = []

    for i in range(2000):
        start = time.perf_counter_ns()
        model.predict(X_test[i:i+1])
        single_times.append(
            (time.perf_counter_ns() - start) / 1_000_000
        )

    mean_latency = float(np.mean(single_times))
    median_latency = float(np.median(single_times))
    p95_latency = float(np.percentile(single_times, 95))

    # 2. Batch inference
    batch_times = []

    for _ in range(5):
        start = time.perf_counter()

        for i in range(0, len(X_test), batch_size):
            model.predict(X_test[i:i+batch_size])

        batch_times.append(time.perf_counter() - start)

    total_time = float(np.median(batch_times))
    throughput = len(X_test) / total_time

    print(f"Mean latency:   {mean_latency:.6f} ms")
    print(f"Median latency: {median_latency:.6f} ms")
    print(f"P95 latency:    {p95_latency:.6f} ms")
    print(f"Batch throughput: {throughput:.2f} flows/s")
    print(f"Total test time:  {total_time:.4f} s")

    return {
        "Model": name,
        "Mean Latency (ms)": mean_latency,
        "Median Latency (ms)": median_latency,
        "P95 Latency (ms)": p95_latency,
        "Throughput (flows/s)": throughput
    }



# Random Forest Inference Benchmark
rf.set_params(n_jobs=1)
rf_single_thread = benchmark_model(
    rf, X_test, "Random Forest"
)

# K-Means Inference Benchmark
kmeans_benchmark = benchmark_model(
    kmeans, X_test, "K-Means"
)

In [ ]:

import pandas as pd
from pathlib import Path

# Φόρτωση αποτελεσμάτων MLP
mlp = pd.read_csv(
    "results/mlp_final_summary.csv"
).iloc[0]

# Συγκέντρωση αποτελεσμάτων
comparison = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Accuracy": rf_metrics["Accuracy"],
        "Precision": rf_metrics["Precision"],
        "Recall": rf_metrics["Recall"],
        "F1": rf_metrics["F1-score"],
        "FPR": rf_metrics["FPR"],
        "Median Latency (ms)": rf_single_thread["Median Latency (ms)"],
        "P95 Latency (ms)": rf_single_thread["P95 Latency (ms)"],
        "Throughput (flows/s)": rf_single_thread["Throughput (flows/s)"]
    },
    {
        "Model": "K-Means",
        "Accuracy": kmeans_metrics["Accuracy"],
        "Precision": kmeans_metrics["Precision"],
        "Recall": kmeans_metrics["Recall"],
        "F1": kmeans_metrics["F1-score"],
        "FPR": kmeans_metrics["FPR"],
        "Median Latency (ms)": kmeans_benchmark["Median Latency (ms)"],
        "P95 Latency (ms)": kmeans_benchmark["P95 Latency (ms)"],
        "Throughput (flows/s)": kmeans_benchmark["Throughput (flows/s)"]
    },
    {
        "Model": "MLP",
        "Accuracy": mlp["Accuracy"],
        "Precision": mlp["Precision"],
        "Recall": mlp["Recall"],
        "F1": mlp["F1"],
        "FPR": mlp["FPR"],
        "Median Latency (ms)": mlp["Median Latency (ms)"],
        "P95 Latency (ms)": mlp["P95 Latency (ms)"],
        "Throughput (flows/s)": mlp["Throughput (flows/s)"]
    }
])

# Αποθήκευση
Path("results").mkdir(exist_ok=True)

comparison.to_csv(
    "results/model_comparison.csv",
    index=False
)

print("=== FINAL MODEL COMPARISON ===")
print(comparison.round(6).to_string(index=False))

print("\nSaved: results/model_comparison.csv")


In [ ]:

import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=== UNSUPERVISED: ISOLATION FOREST ===")

# 1. Δημιουργία μοντέλου
iso_forest = IsolationForest(
    n_estimators=100,
    max_samples="auto",
    contamination=0.10,
    random_state=42,
    n_jobs=-1
)

# 2. Εκπαίδευση χωρίς Labels
iso_forest.fit(X_train)

print("Isolation Forest trained successfully!")

# 3. Πρόβλεψη στο Testing Set
raw_predictions = iso_forest.predict(X_test)

# Isolation Forest:
#  1 = Normal
# -1 = Anomaly

# Το δικό μας binary classification:
# 0 = Benign
# 1 = Attack
iso_preds = np.where(
    raw_predictions == -1, 1, 0
)

# 4. Αξιολόγηση
accuracy = accuracy_score(y_test_bin, iso_preds)

precision = precision_score(
    y_test_bin, iso_preds, zero_division=0
)

recall = recall_score(
    y_test_bin, iso_preds, zero_division=0
)

f1 = f1_score(
    y_test_bin, iso_preds, zero_division=0
)

cm = confusion_matrix(
    y_test_bin, iso_preds, labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn)

print("\n=== ISOLATION FOREST RESULTS ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"FPR:       {fpr:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nPredicted anomalies:", int(iso_preds.sum()))
print(
    "Predicted anomaly percentage:",
    round(100 * iso_preds.mean(), 2),
    "%"
)
